# Enginuity Dataset — Global Manifest Analysis
Analysis of figure-to-CSV mappings across all processed PDFs.

In [15]:
import pandas as pd

# ── CONFIG ──────────────────────────────────────────────────────────────────
MANIFEST_PATH = "/opt/predii/isha/enginuity-bench/results/dataset_manifest_global_vllm.tsv"
# ────────────────────────────────────────────────────────────────────────────

df = pd.read_csv(MANIFEST_PATH, sep="\t", dtype=str)
print(f"Manifest loaded: {len(df):,} total rows")
print(f"Columns: {list(df.columns)}")
df.head(3)

Manifest loaded: 3,397 total rows
Columns: ['figure_path', 'table_path', 'csv_path', 'figure', 'pdf_id']


,figure_path,table_path,csv_path,figure,pdf_id
0,data-1/figures/page_660.png,data-1/tables/page_661.png,data-1/table_data/page_661/page_661.csv,179,data-1
1,data-1/figures/page_660.png,data-1/tables/page_662.png,data-1/table_data/page_662/page_662.csv,179,data-1
2,data-1/figures/page_660.png,data-1/tables/page_663.png,data-1/table_data/page_663/page_663.csv,179,data-1


### Unique figures with at least one CSV mapped

In [16]:
# Keep only rows where csv_path is present
has_csv = df[df["csv_path"].notna() & (df["csv_path"].str.strip() != "")]

unique_figures_with_csv = has_csv["figure_path"].nunique()
total_unique_figures    = df["figure_path"].nunique()
figures_without_csv     = total_unique_figures - unique_figures_with_csv

print("=" * 45)
print(f"  Total unique figures          : {total_unique_figures:>6,}")
print(f"  Figures WITH a csv_path       : {unique_figures_with_csv:>6,}")
print(f"  Figures WITHOUT a csv_path    : {figures_without_csv:>6,}")
print("=" * 45)

# Per-PDF breakdown
print("\nPer PDF:")
per_pdf = (
    has_csv.groupby("pdf_id")["figure_path"]
    .nunique()
    .reset_index()
    .rename(columns={"figure_path": "unique_figures_with_csv"})
    .sort_values("pdf_id")
)
display(per_pdf)

  Total unique figures          :  2,006
  Figures WITH a csv_path       :  2,006
  Figures WITHOUT a csv_path    :      0

Per PDF:


,pdf_id,unique_figures_with_csv
0,data-1,341
1,data-10,171
2,data-2,350
3,data-3,269
4,data-4,298
5,data-5,61
6,data-6,51
7,data-7,33
8,data-8,92
9,data-9,340


### Number of CSV paths per figure (multi-table analysis)

In [17]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Count CSVs per figure
csv_per_fig = (
    has_csv.groupby("figure_path")["csv_path"]
    .count()
    .reset_index()
    .rename(columns={"csv_path": "csv_count"})
)

# ── Summary statistics ───────────────────────────────────────────────────────
stats = csv_per_fig["csv_count"].describe().rename("csv_count_per_figure")
print("Summary statistics — CSVs per figure:")
print(stats.to_string())
print()

# Distribution table
dist = csv_per_fig["csv_count"].value_counts().sort_index().reset_index()
dist.columns = ["csv_count", "num_figures"]
dist["pct"] = (dist["num_figures"] / dist["num_figures"].sum() * 100).round(1)
dist["label"] = dist["csv_count"].apply(
    lambda x: f"{x} CSV{'s' if x > 1 else ''} (single table)" if x == 1 else f"{x} CSVs (multi-table)"
)

print("Distribution of CSV count per figure:")
display(dist[["csv_count", "num_figures", "pct", "label"]])

single = dist.loc[dist["csv_count"] == 1, "num_figures"].sum()
multi  = dist.loc[dist["csv_count"]  > 1, "num_figures"].sum()
print(f"\nSingle-table figures : {single:,} ({single/len(csv_per_fig)*100:.1f}%)")
print(f"Multi-table figures  : {multi:,}  ({multi/len(csv_per_fig)*100:.1f}%)")

Summary statistics — CSVs per figure:
count    2006.000000
mean        1.693420
std         0.944456
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max         5.000000

Distribution of CSV count per figure:


,csv_count,num_figures,pct,label
0,1,1106,55.1,1 CSV (single table)
1,2,572,28.5,2 CSVs (multi-table)
2,3,192,9.6,3 CSVs (multi-table)
3,4,109,5.4,4 CSVs (multi-table)
4,5,27,1.3,5 CSVs (multi-table)



Single-table figures : 1,106 (55.1%)
Multi-table figures  : 900  (44.9%)


### Threshold filter: figures available at a given max CSV count

In [18]:
# ── CONFIG ──────────────────────────────────────────────────────────────────
THRESHOLD = 5
# ────────────────────────────────────────────────────────────────────────────

filtered = dist[dist["csv_count"] <= THRESHOLD]
total_figures = filtered["num_figures"].sum()
print(f"Figures available with at most {THRESHOLD} CSV(s): {total_figures:,}")

Figures available with at most 5 CSV(s): 2,006
